# TAREA: 📊 Análisis con Regresión Lineal

En esta tarea se utilizará un **modelo de regresión lineal** para predecir la variable **`YRayosUV`** (nivel de radiación ultravioleta).  

La base de datos cuenta con **1551 registros** y **13 variables**, entre las que se incluyen factores climáticos como **temperatura, humedad, velocidad del viento, radiación solar, energía solar y sensación térmica**.  

In [1]:
# =========================================
# 1) Cargar datos y objetivo (tu dataset)
# =========================================
import os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_FILE  = "datos_rayos_uv.xlsx"   # <-- tu archivo
SHEET_NAME = "reporte2019"                   # <-- hoja
TARGET     = "YRayosUV"                      # <-- variable objetivo

assert os.path.exists(DATA_FILE), f"No se encuentra {DATA_FILE}"

# Leer Excel
df = pd.read_excel(DATA_FILE, sheet_name=SHEET_NAME)

# Inspección
df.info()

# Definir X (predictoras) e y (objetivo)
y = df[TARGET]
X = df.drop(columns=[TARGET])

print(
    "Shape X:", X.shape,
    "| y(mean):", round(y.mean(), 6),
    "| y(std):", round(y.std(), 6),
    "| y[min,max]:", (round(y.min(), 6), round(y.max(), 6))
)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1551 entries, 0 to 1550
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   YRayosUV     1551 non-null   float64
 1   X1Temamb     1551 non-null   float64
 2   X2Hum        1551 non-null   int64  
 3   X3Velviento  1551 non-null   float64
 4   X4VieFrío    1551 non-null   float64
 5   X5Incalor    1551 non-null   float64
 6   X6IndTHW     1551 non-null   float64
 7   X7Radsolar   1551 non-null   int64  
 8   X8EneSolar   1551 non-null   float64
 9   X9Temp       1551 non-null   float64
 10  X10Humed     1551 non-null   int64  
 11  X11Rocío     1551 non-null   float64
 12  X12Calor     1551 non-null   float64
dtypes: float64(10), int64(3)
memory usage: 157.7 KB
Shape X: (1551, 12) | y(mean): 0.181786 | y(std): 0.118598 | y[min,max]: (0.0, 0.41)


In [2]:
X , y

(      X1Temamb  X2Hum  X3Velviento  X4VieFrío  X5Incalor  X6IndTHW  \
 0         28.5     39          8.0       28.5       28.3      28.3   
 1         28.4     40          8.0       28.4       28.3      28.3   
 2         28.2     40         11.3       28.2       28.0      28.0   
 3         28.3     40          9.7       28.3       28.2      28.2   
 4         28.4     41          8.0       28.4       28.4      28.4   
 ...        ...    ...          ...        ...        ...       ...   
 1546      27.6     42          8.0       27.6       27.2      27.2   
 1547      27.7     42          6.4       27.7       27.4      27.4   
 1548      27.7     41          6.4       27.7       27.3      27.3   
 1549      27.5     41          9.7       27.5       27.1      27.1   
 1550      27.4     42          9.7       27.4       27.1      27.1   
 
       X7Radsolar  X8EneSolar  X9Temp  X10Humed  X11Rocío  X12Calor  
 0            790        5.66    28.7        58      19.6      30.4  
 1    

In [3]:
# =========================================
# 2) Split temprano (80/20)
# =========================================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20,  random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (1240, 12) | Test: (311, 12)


**🔹 División de datos (Train/Test)**

El dataset se dividió en dos subconjuntos para poder entrenar y evaluar el modelo de forma objetiva:

- **Entrenamiento (Train):** 1240 registros con 12 variables predictoras.  
- **Prueba (Test):** 311 registros con las mismas 12 variables predictoras.  

Esta división corresponde a un **80% para entrenamiento** y un **20% para prueba**, lo que permite que el modelo aprenda con la mayor parte de los datos y luego se valide con información que no ha visto previamente.


In [4]:
# =========================================
# 3) Preprocesamiento (en pipeline)
# =========================================
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from imblearn.pipeline import Pipeline as ImbPipeline  # imblearn solo por consistencia de API

cat_features = X_train.select_dtypes(include=["object","category"]).columns.tolist()
num_features = X_train.select_dtypes(include=["number","bool"]).columns.tolist()

# OneHotEncoder compatible (con fallback)
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", ohe,              cat_features),
    ],
    remainder="drop",
)

def build_pipe(model):
    # Nota: en regresión NO se usa SMOTE
    return ImbPipeline([
        ("prep", preprocessor),
        ("var0", VarianceThreshold(0.0)),  # limpia columnas constantes tras OHE
        ("model", model),
    ])
    
print(f"Features numéricas: {num_features}")
print(f"Features categóricas: {cat_features}")

Features numéricas: ['X1Temamb', 'X2Hum', 'X3Velviento', 'X4VieFrío', 'X5Incalor', 'X6IndTHW', 'X7Radsolar', 'X8EneSolar', 'X9Temp', 'X10Humed', 'X11Rocío', 'X12Calor']
Features categóricas: []


In [7]:
# =========================================
# 4) Modelos candidatos (REGRESIÓN)
# =========================================
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

candidates = [
    ("LR",  LinearRegression()),
    ("RG",  Ridge(random_state=RANDOM_STATE)),
    ("LS",  Lasso(random_state=RANDOM_STATE, max_iter=5000)),
    ("EN",  ElasticNet(random_state=RANDOM_STATE, max_iter=5000)),
    ("KNR", KNeighborsRegressor()),
    ("DTR", DecisionTreeRegressor(random_state=RANDOM_STATE)),
    ("RFR", RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
    ("MLP", MLPRegressor(hidden_layer_sizes=(64,), max_iter=800, random_state=RANDOM_STATE)),
    ("XGB", XGBRegressor(tree_method="hist", random_state=RANDOM_STATE,
                         n_estimators=400, learning_rate=0.05, max_depth=6,
                         subsample=0.9, colsample_bytree=0.9, n_jobs=-1)),
    ("LGB", LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=-1,
                          subsample=0.9, colsample_bytree=0.9,
                          random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1)),
    ("CAT", CatBoostRegressor(iterations=600, learning_rate=0.05, depth=6,
                              random_state=RANDOM_STATE, l2_leaf_reg=3.0,
                              verbose=False, allow_writing_files=False, thread_count=-1)),
]

In [8]:
# =========================================
# 5) Baseline con CV (sin tuning)
# =========================================
from sklearn.model_selection import KFold, cross_validate
import pandas as pd

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae":  "neg_mean_absolute_error",
    "r2":   "r2",
}

rows = []
for name, model in candidates:
    pipe = build_pipe(model)
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    row = {
        "model": name,
        "rmse": -scores["test_rmse"].mean(),
        "mae":  -scores["test_mae"].mean(),
        "r2":    scores["test_r2"].mean(),
    }
    rows.append(row)
    print(f"{name:>3} | RMSE {row['rmse']:.3f} | MAE {row['mae']:.3f} | R² {row['r2']:.3f}")

baseline_df = pd.DataFrame(rows).sort_values("rmse")
display(baseline_df)
baseline_best_name  = baseline_df.iloc[0]["model"]
baseline_best_model = dict(candidates)[baseline_best_name]
print(f">>> Baseline ganador: {baseline_best_name}")

 LR | RMSE 0.032 | MAE 0.025 | R² 0.926
 RG | RMSE 0.032 | MAE 0.025 | R² 0.926
 LS | RMSE 0.118 | MAE 0.105 | R² -0.007
 EN | RMSE 0.118 | MAE 0.105 | R² -0.007
KNR | RMSE 0.021 | MAE 0.015 | R² 0.968
DTR | RMSE 0.026 | MAE 0.017 | R² 0.950
RFR | RMSE 0.018 | MAE 0.012 | R² 0.977
MLP | RMSE 0.055 | MAE 0.042 | R² 0.778
XGB | RMSE 0.016 | MAE 0.011 | R² 0.982
LGB | RMSE 0.016 | MAE 0.011 | R² 0.982
CAT | RMSE 0.015 | MAE 0.011 | R² 0.983


,model,rmse,mae,r2
10,CAT,0.015424,0.010772,0.982709
8,XGB,0.015853,0.010802,0.981705
9,LGB,0.015856,0.011024,0.981720
6,RFR,0.017763,0.012146,0.977053
4,KNR,0.020993,0.015371,0.967948
5,DTR,0.026142,0.016621,0.950317
0,LR,0.031854,0.025202,0.926069
1,RG,0.031962,0.025337,0.925601
7,MLP,0.055200,0.041761,0.778057
2,LS,0.117894,0.105175,-0.007077


>>> Baseline ganador: CAT


In [9]:
# =========================================
# 6) Tuning con CV y elección del ganador (rápido)
# =========================================
import tempfile, shutil
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
try:
    from scipy.stats import loguniform
except Exception:
    from sklearn.utils.fixes import loguniform

cv_light = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_heavy = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

param_spaces = {
    "RG":  {"model__alpha": loguniform(1e-3, 1e3)},
    "LS":  {"model__alpha": loguniform(1e-3, 1e2)},
    "EN":  {"model__alpha": loguniform(1e-3, 1e2), "model__l1_ratio": uniform(0.0, 1.0)},
    "KNR": {"model__n_neighbors": randint(2, 50), "model__weights": ["uniform","distance"], "model__p":[1,2]},
    "DTR": {"model__max_depth": randint(3, 16), "model__min_samples_leaf": randint(1, 10)},
    "RFR": {"model__n_estimators": randint(200, 600), "model__max_depth": randint(4, 16),
            "model__min_samples_split": randint(2, 20), "model__min_samples_leaf": randint(1, 10),
            "model__max_features": ["sqrt","log2", None], "model__bootstrap": [True, False]},
    "MLP": {"model__alpha": loguniform(1e-4, 1e-1), "model__learning_rate_init": loguniform(1e-4, 1e-2)},
    "XGB": {"model__n_estimators": randint(250, 600), "model__learning_rate": loguniform(5e-3, 2e-1),
            "model__max_depth": randint(3, 9), "model__subsample": uniform(0.7, 0.3),
            "model__colsample_bytree": uniform(0.7, 0.3), "model__min_child_weight": randint(1, 6)},
    "LGB": {"model__n_estimators": randint(300, 800), "model__learning_rate": loguniform(5e-3, 2e-1),
            "model__num_leaves": randint(16, 128), "model__max_depth": randint(-1, 12),
            "model__min_child_samples": randint(10, 50), "model__subsample": uniform(0.7, 0.3),
            "model__colsample_bytree": uniform(0.7, 0.3), "model__reg_lambda": loguniform(1e-3, 10)},
    "CAT": {"model__iterations": randint(300, 700), "model__learning_rate": loguniform(5e-3, 2e-1),
            "model__depth": randint(4, 10), "model__l2_leaf_reg": loguniform(1e-2, 30),
            "model__border_count": randint(32, 255)},
}

to_tune = [
    ("RG",  Ridge(random_state=RANDOM_STATE)),
    ("EN",  ElasticNet(random_state=RANDOM_STATE, max_iter=5000)),
    ("RFR", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1)),
    ("XGB", XGBRegressor(tree_method="hist", random_state=RANDOM_STATE, n_jobs=1)),
    ("LGB", LGBMRegressor(random_state=RANDOM_STATE, n_jobs=1, verbosity=-1)),
    ("CAT", CatBoostRegressor(random_state=RANDOM_STATE, verbose=False, allow_writing_files=False, thread_count=1)),
]

refit_metric = "rmse"  # minimizamos RMSE
scoring = {"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"}

best_models = []
cache_dir = tempfile.mkdtemp(prefix="skcache_")
try:
    for name, base_model in to_tune:
        pipe = build_pipe(base_model)
        try: pipe.set_params(memory=cache_dir)
        except: pass
        heavy = name in ["RFR","XGB","LGB","CAT"]
        search = RandomizedSearchCV(
            pipe, param_spaces[name],
            n_iter=(15 if heavy else 12),
            cv=(cv_heavy if heavy else cv_light),
            scoring=scoring, refit="rmse",
            n_jobs=-1, random_state=RANDOM_STATE, verbose=1,
            error_score=np.nan, return_train_score=False
        )
        search.fit(X_train, y_train)
        best_models.append((name, search.best_estimator_, -search.best_score_, search.best_params_))  # RMSE positivo
    best_models.sort(key=lambda x: x[2])  # menor RMSE primero
    best_name, final_pipe_opt, best_cv_rmse, best_params = best_models[0]
    print(f">>> GANADOR OPTIMIZADO: {best_name} (RMSE CV={best_cv_rmse:.3f})")
finally:
    shutil.rmtree(cache_dir, ignore_errors=True)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Fitting 3 folds for each of 15 candidates, totalling 45 fits
Fitting 3 folds for each of 15 candidates, totalling 45 fits
Fitting 3 folds for each of 15 candidates, totalling 45 fits
Fitting 3 folds for each of 15 candidates, totalling 45 fits
>>> GANADOR OPTIMIZADO: CAT (RMSE CV=0.016)


In [10]:
# =========================================
# 7) Comparación justa (solo CV) - baseline vs ganador
# =========================================
from sklearn.model_selection import KFold, cross_validate

same_cv = KFold(n_splits=5, shuffle=True, random_state=123)
pipe_baseline_best = build_pipe(baseline_best_model)
pipe_tuned_best    = final_pipe_opt

def cv_rmse(pipe, name):
    s = cross_validate(pipe, X_train, y_train, cv=same_cv,
                       scoring={"rmse":"neg_root_mean_squared_error"}, n_jobs=-1)
    rmse = -s["test_rmse"].mean()
    print(f"{name}: RMSE {rmse:.4f}")
    return rmse

rmse_base = cv_rmse(pipe_baseline_best, f"Baseline({baseline_best_name})")
rmse_tune = cv_rmse(pipe_tuned_best,   f"Tuned({best_name})")

# Regla: si la mejora < 1% del RMSE base, nos quedamos con el baseline (más simple)
if (rmse_base - rmse_tune) / rmse_base >= 0.01:
    winner_name, winner_pipe = best_name, pipe_tuned_best
else:
    winner_name, winner_pipe = baseline_best_name, pipe_baseline_best

print(f">>> Modelo seleccionado para TEST: {winner_name}")


Baseline(CAT): RMSE 0.0158
Tuned(CAT): RMSE 0.0156
>>> Modelo seleccionado para TEST: CAT


In [11]:
# =========================================
# 8) Política de decisión (mínima)
# =========================================
POLICY = {
    "clip_to_train_range": True,   # recorta predicciones al rango visto en TRAIN
    "round_to_int": False,         # pon True si el objetivo es entero (conteos)
    "lower": float(y_train.min()),
    "upper": float(y_train.max()),
}
print("Política:", POLICY)

def postprocess_preds(yhat, policy=POLICY):
    ypp = yhat.copy()
    if policy.get("clip_to_train_range", False):
        ypp = np.clip(ypp, policy["lower"], policy["upper"])
    if policy.get("round_to_int", False):
        ypp = np.rint(ypp).astype(int)
    return ypp

Política: {'clip_to_train_range': True, 'round_to_int': False, 'lower': 0.0, 'upper': 0.4}


In [12]:
# =========================================
# 9) Evaluación final en TEST
# =========================================
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

winner_pipe.fit(X_train, y_train)
y_pred = winner_pipe.predict(X_test)
y_pp   = postprocess_preds(y_pred, POLICY)

rmse = mean_squared_error(y_test, y_pp, squared=False)
mae  = mean_absolute_error(y_test, y_pp)
r2   = r2_score(y_test, y_pp)

print(f"TEST → RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")

# vistazo rápido (primeros 10)
import pandas as pd
preview = pd.DataFrame({"y_true": y_test.reset_index(drop=True),
                        "y_pred": pd.Series(y_pp)}).head(10)
print(preview.to_string(index=False))


TEST → RMSE: 0.0141 | MAE: 0.0094 | R²: 0.9865
 y_true   y_pred
   0.24 0.235881
   0.06 0.061189
   0.31 0.308761
   0.32 0.309210
   0.12 0.117009
   0.25 0.256162
   0.32 0.315433
   0.29 0.244096
   0.25 0.184742
   0.08 0.075145


In [15]:
# =========================================
# 10) Interpretabilidad + breve error analysis (mínimo, FIX)
# =========================================
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error

# Asegurar que existe un modelo ganador entrenado
try:
    winner_pipe
except NameError:
    winner_pipe = build_pipe(baseline_best_model)
    winner_pipe.fit(X_train, y_train)

# Política simple: recortar predicciones al rango observado en entrenamiento
POLICY = {
    "lower": float(y_train.min()),
    "upper": float(y_train.max()),
}
def postprocess_preds(yhat, policy):
    return np.clip(yhat, policy["lower"], policy["upper"])

# 10.1 ¿Cuánto recorta la política?
raw_pred  = winner_pipe.predict(X_test)
clip_low  = (raw_pred < POLICY["lower"]).mean()
clip_high = (raw_pred > POLICY["upper"]).mean()
print(f"[Policy] clipped_low: {clip_low:.3%} | clipped_high: {clip_high:.3%}")

# 10.2 Importancias por Permutación (sobre columnas ORIGINALES)
r = permutation_importance(
    winner_pipe,
    X_test, y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    scoring="neg_root_mean_squared_error"
)

feat_names = X_test.columns
imp = (pd.DataFrame({
        "feature": feat_names,
        "importance": r.importances_mean,
        "std": r.importances_std
     })
     .sort_values("importance", ascending=False)
     .head(15)
)
print("\nTop-15 importancias (perm, columnas originales):")
print(imp.to_string(index=False))

# 10.3 Errores: resumen + peores casos
y_hat = winner_pipe.predict(X_test)
y_pp  = postprocess_preds(y_hat, POLICY)
res   = pd.DataFrame({
    "y_true": y_test.reset_index(drop=True),
    "y_pred": pd.Series(y_pp)
})
res["abs_err"] = (res["y_true"] - res["y_pred"]).abs()
print("\nResumen de |error|:")
print(res["abs_err"].describe(percentiles=[.1,.25,.5,.75,.9]).to_string())

print("\nPeores 10 casos (|error| alto):")
top_bad_idx = res["abs_err"].nlargest(10).index
print(pd.concat([res.loc[top_bad_idx], X_test.reset_index(drop=True).loc[top_bad_idx]], axis=1)
      .to_string(index=False))

# 10.4 Métricas por subgrupos (no aplica si no hay variables categóricas)


[Policy] clipped_low: 0.000% | clipped_high: 0.322%

Top-15 importancias (perm, columnas originales):
    feature  importance      std
 X8EneSolar    0.061645 0.002169
 X7Radsolar    0.046996 0.001853
   X10Humed    0.026481 0.001596
   X12Calor    0.011428 0.000431
     X9Temp    0.008816 0.000616
  X5Incalor    0.007872 0.000555
   X6IndTHW    0.005625 0.000361
X3Velviento    0.005505 0.000634
      X2Hum    0.004585 0.000468
   X1Temamb    0.004397 0.000414
   X11Rocío    0.004379 0.000530
  X4VieFrío    0.002810 0.000278

Resumen de |error|:
count    311.000000
mean       0.009450
std        0.010527
min        0.000000
10%        0.001117
25%        0.002997
50%        0.006273
75%        0.012002
90%        0.020740
max        0.075911

Peores 10 casos (|error| alto):
 y_true   y_pred  abs_err  X1Temamb  X2Hum  X3Velviento  X4VieFrío  X5Incalor  X6IndTHW  X7Radsolar  X8EneSolar  X9Temp  X10Humed  X11Rocío  X12Calor
   0.25 0.325911 0.075911      26.9     46          9.7       26.

In [16]:
Xtr = winner_pipe.named_steps["prep"].transform(X_test)
model = winner_pipe.named_steps["model"]
r2 = permutation_importance(model, Xtr, y_test, n_repeats=10,
                            random_state=RANDOM_STATE,
                            scoring="neg_root_mean_squared_error")
feat_names_ohe = winner_pipe.named_steps["prep"].get_feature_names_out()
imp_ohe = pd.DataFrame({"feature": feat_names_ohe,
                        "importance": r2.importances_mean,
                        "std": r2.importances_std}).sort_values("importance", ascending=False).head(20)


In [ ]:
imp_ohe